### Extra: Stress-Testing the Model with Harder Questions
Dina Bosma-Buczynska  

**LangSmith experiment:** [gpt-4o-mini-hard-9af36d57](https://eu.smith.langchain.com/o/561aa291-0a5c-4c0b-aad9-85d2023d43c6/datasets/b2ecbd10-70d1-49e0-90b7-fb9c1ececbf1/compare?selectedSessions=ae50092c-ef50-44db-9e47-6ef8942d01e5)

---

#### Why this notebook exists

The main lab (LAB7.03) produced a ceiling result: `gpt-4o-mini` scored **95% correctness**
and `gpt-4o` scored **100%** on 20 digital marketing analytics questions.

That is not purely a success. A near-perfect score on well-formed, unambiguous questions
tells you the model knows its definitions — but it does not tell you whether the model
can handle the messier queries that show up in real analyst workflows:

- Questions with no single correct answer (context-dependent)
- Conflicting metrics that require diagnostic reasoning
- Situations where the honest answer is *"your sample is too small to conclude anything"*
- Questions built on a flawed premise that the model should push back on
- Multi-step reasoning that requires combining several concepts

**The hypothesis tested here:**  
A dataset that only contains clean, lookup-style questions will always score near 1.0
regardless of model quality. To make the evaluation meaningful, you need questions that
punish overconfidence and reward nuance. This notebook tests that hypothesis by running
the same model and evaluators against 12 deliberately harder examples and comparing the results.


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"]  = "lab7-marketing-qa-eval"

from langsmith import Client
from langsmith.wrappers import wrap_openai
from langsmith import traceable
from openai import OpenAI
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import json as _json
import collections

plt.style.use("seaborn-v0_8")

client        = Client()
openai_client = wrap_openai(OpenAI())
print("LangSmith connection OK")


/Users/dinabosmabuczynska/Desktop/bootcamp_env/.conda/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


LangSmith connection OK


In [2]:
# Replicated from main lab so this notebook runs standalone

SYSTEM_PROMPT = """You are a senior digital marketing analyst.
Answer the question concisely and accurately.
Include relevant KPI formulas, benchmarks, or action steps where appropriate.
Keep your answer under 150 words."""

@traceable(name="marketing-qa-gpt4o-mini")
def target_gpt4o_mini(inputs: dict) -> dict:
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": inputs["question"]},
        ],
        temperature=0,
        max_tokens=300,
    )
    return {"answer": response.choices[0].message.content.strip()}

# Correctness evaluator
_correctness_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    model="openai:gpt-4o-mini",
    feedback_key="correctness",
)

def correctness_evaluator(inputs, outputs, reference_outputs):
    return _correctness_judge(inputs=inputs, outputs=outputs, reference_outputs=reference_outputs)

# Actionability evaluator
ACTIONABILITY_SYSTEM = """You are evaluating whether a marketing analyst's answer is actionable.

An actionable answer:
- Gives specific steps, thresholds, or decisions the reader can act on
- Includes relevant formulas, benchmarks, or criteria where applicable
- Avoids vague generalities like 'it depends' without follow-up specifics

Score from 0.0 to 1.0:
- 1.0: Highly actionable — clear steps or criteria provided
- 0.7: Mostly actionable — some specifics but could be more concrete
- 0.4: Partially actionable — correct but too general
- 0.1: Not actionable — vague or theoretical only

Respond with valid JSON only: {"score": <float>, "reasoning": "<one sentence>"}"""

@traceable(name="actionability-judge")
def actionability_evaluator(inputs, outputs, reference_outputs=None):
    user_msg = f"Question: {inputs.get('question', '')}\nAnswer: {outputs.get('answer', '')}"
    resp = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": ACTIONABILITY_SYSTEM},
            {"role": "user",   "content": user_msg},
        ],
        temperature=0,
        max_tokens=80,
        response_format={"type": "json_object"},
    )
    result = _json.loads(resp.choices[0].message.content)
    return {"key": "actionability", "score": result["score"], "comment": result["reasoning"]}

print("Target function and evaluators ready.")


Target function and evaluators ready.


---
### The 12 harder questions

Each question is designed to require something beyond definition recall.
Reference answers explicitly include the nuance expected from the model —
acknowledging ambiguity, pushing back on flawed premises, or flagging insufficient data.


In [3]:
hard_examples = [
    # ── Ambiguous — correct answer must acknowledge context dependency ─────
    {
        "question": "Is a CTR of 2% good?",
        "answer": (
            "It depends on the channel and objective. 2% is strong for display "
            "(typical 0.1-0.3%) but weak for branded search (typical 5-15%). "
            "For social feed ads 1-3% is average. CTR is not a verdict on its own — "
            "it must be read against channel benchmarks and paired with conversion rate."
        ),
        "category": "Ambiguous",
    },
    {
        "question": "What is the ideal ROAS for a campaign?",
        "answer": (
            "There is no universal ideal ROAS. Break-even ROAS = 1 / gross margin. "
            "A 40% margin business needs ROAS > 2.5 just to cover product cost before overhead. "
            "Target ROAS must be set per business, channel, and objective. "
            "An awareness campaign should not be judged on ROAS at all."
        ),
        "category": "Ambiguous",
    },
    {
        "question": "Bounce rate increased from 40% to 55% after a landing page redesign. Is this a problem?",
        "answer": (
            "Not necessarily. Bounce rate depends on page intent — a blog or contact page "
            "with high bounce can be a success if users got what they needed. "
            "Check whether conversion rate, time-on-page, or revenue changed. "
            "Diagnose with scroll depth, heatmaps, and funnel data before concluding the redesign hurt."
        ),
        "category": "Ambiguous",
    },
    # ── Conflicting signals — requires diagnostic reasoning ───────────────
    {
        "question": "A campaign has CPA $12 but ROAS 0.8. Both are reported KPIs. How do you interpret this?",
        "answer": (
            "The conflict signals a revenue-per-conversion problem, not an acquisition issue. "
            "Customers are acquired cheaply ($12) but each generates less than $1 revenue per $1 spend. "
            "Likely causes: low AOV, high return rate, or misattribution. "
            "Investigate revenue tracking completeness. Do not optimise CPA further — unit economics are broken upstream."
        ),
        "category": "Conflicting",
    },
    {
        "question": "Impression share is 95% but ROAS dropped 30% last month. What do you investigate first?",
        "answer": (
            "High impression share rules out budget or bid constraints. Investigate: "
            "(1) conversion tracking — broken pixels or attribution window changes; "
            "(2) audience quality shift; (3) landing page conversion rate; "
            "(4) competitor offer changes; (5) seasonality. "
            "Do not increase bids — that would worsen ROAS further."
        ),
        "category": "Conflicting",
    },
    {
        "question": "Variant B has 12% higher conversion rate but p=0.07 after 3 weeks with 800 visitors per variant. What do you recommend?",
        "answer": (
            "Do not call this significant at alpha=0.05, but do not dismiss it. "
            "p=0.07 with 800 visitors suggests underpowered test — ~1,500-2,000 needed for 80% power. "
            "Recommendation: extend the test rather than deciding now. "
            "Calling it early risks a false negative or false positive."
        ),
        "category": "Conflicting",
    },
    # ── Insufficient data — correct answer flags sample size ──────────────
    {
        "question": "A new channel had 4 conversions this week at CPA $8. Target CPA is $25. Should you scale immediately?",
        "answer": (
            "No. 4 conversions is statistically meaningless — confidence intervals are extremely wide. "
            "The $8 CPA could be luck, early cherry-picked converters, or a tracking anomaly. "
            "Run 2-3 weeks to accumulate 30-50 conversions, verify tracking, "
            "and check for audience saturation before scaling."
        ),
        "category": "Insufficient data",
    },
    {
        "question": "You compared CPA between two channels and got p=0.04 with n=12 total conversions. Can you conclude one is more efficient?",
        "answer": (
            "No. With ~6 conversions per channel the test is severely underpowered. "
            "A p-value below 0.05 from 6 events per group is more likely a Type I error than a real effect. "
            "Collect at least 30-50 conversions per channel before running significance tests. "
            "Always report sample size alongside p-values."
        ),
        "category": "Insufficient data",
    },
    # ── Trick / premise challenge — model should push back ────────────────
    {
        "question": "An awareness campaign has conversion rate 0.1% and ROAS 0.3. It is clearly underperforming. How do you fix the ROAS?",
        "answer": (
            "The premise is flawed. An awareness campaign should not be evaluated on ROAS or conversion rate — "
            "these are lower-funnel metrics. Awareness is correctly measured on reach, frequency, brand lift, and CPM. "
            "Optimising for ROAS will narrow the audience to warm users, defeating the purpose. "
            "The fix is to apply appropriate KPIs per funnel stage, not to change the campaign."
        ),
        "category": "Trick",
    },
    {
        "question": "After Bonferroni correction your significant result (p=0.03) becomes p=0.09. You should treat the channels as identical. Agree?",
        "answer": (
            "Not necessarily. Bonferroni is conservative and may be too strict here. "
            "Check Cohen's d — if effect size is medium or large the result warrants investigation "
            "even below the corrected threshold. Consider Benjamini-Hochberg FDR as an alternative. "
            "Non-significance after correction is not proof of equivalence."
        ),
        "category": "Trick",
    },
    # ── Multi-step reasoning ──────────────────────────────────────────────
    {
        "question": "Channel A: ROAS 4.2, spend $50K, CPA $18. Channel B: ROAS 1.9, spend $8K, CPA $42. You have $15K extra. How do you allocate?",
        "answer": (
            "Do not move all $15K to Channel A automatically. "
            "First check whether Channel A shows diminishing returns — ROAS degrades as audiences exhaust. "
            "If it absorbs spend without CPA rising above target, allocate $10-12K there. "
            "Retain $3-5K for Channel B if the audience differs and CPA is below customer LTV. "
            "Review after 2-3 weeks with statistical checks before permanent reallocation."
        ),
        "category": "Multi-step",
    },
    {
        "question": "A B2B niche campaign has frequency 11 and CTR 3.2%. Standard guidance says pause above frequency 5. Should you pause?",
        "answer": (
            "Not necessarily. Frequency thresholds of 3-5 apply to B2C mass audiences. "
            "In B2B niche targeting, frequency 8-12 is often needed for message recall. "
            "The key signal is CTR trend: if CTR holds at 3.2% despite frequency 11, the audience is not fatigued. "
            "Monitor week-over-week and refresh creative if CTR starts declining before pausing."
        ),
        "category": "Multi-step",
    },
]

print(f"Hard examples: {len(hard_examples)}")
print("By category:", dict(collections.Counter(e["category"] for e in hard_examples)))


Hard examples: 12
By category: {'Ambiguous': 3, 'Conflicting': 3, 'Insufficient data': 2, 'Trick': 2, 'Multi-step': 2}


In [4]:
HARD_DATASET_NAME = "marketing-analytics-qa-hard-DinaBB"

existing_names = [d.name for d in client.list_datasets()]

if HARD_DATASET_NAME not in existing_names:
    hard_dataset = client.create_dataset(
        HARD_DATASET_NAME,
        description=(
            "12 stress-test digital marketing analytics Q&A examples covering ambiguous, "
            "conflicting-signal, insufficient-data, trick, and multi-step reasoning questions. "
            "Designed to expose model failure modes not visible in the easy dataset."
        ),
    )
    client.create_examples(
        inputs=[{"question": e["question"], "category": e["category"]} for e in hard_examples],
        outputs=[{"answer": e["answer"]} for e in hard_examples],
        dataset_id=hard_dataset.id,
    )
    print(f"Created '{HARD_DATASET_NAME}' with {len(hard_examples)} examples.")
else:
    print(f"'{HARD_DATASET_NAME}' already exists — skipping upload.")
    hard_dataset = next(d for d in client.list_datasets() if d.name == HARD_DATASET_NAME)

print("Dataset ID:", hard_dataset.id)


'marketing-analytics-qa-hard-DinaBB' already exists — skipping upload.


Dataset ID: b2ecbd10-70d1-49e0-90b7-fb9c1ececbf1


In [5]:
results_hard = client.evaluate(
    target_gpt4o_mini,
    data=HARD_DATASET_NAME,
    evaluators=[correctness_evaluator, actionability_evaluator],
    experiment_prefix="gpt-4o-mini-hard",
    max_concurrency=2,
)
print("Hard dataset evaluation complete.")


/Users/dinabosmabuczynska/Desktop/bootcamp_env/.conda/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'gpt-4o-mini-hard-9af36d57' at:
https://eu.smith.langchain.com/o/561aa291-0a5c-4c0b-aad9-85d2023d43c6/datasets/b2ecbd10-70d1-49e0-90b7-fb9c1ececbf1/compare?selectedSessions=ae50092c-ef50-44db-9e47-6ef8942d01e5





0it [00:00, ?it/s]


1it [00:10, 10.64s/it]


3it [00:15,  4.45s/it]

...


10it [00:35,  2.94s/it]


11it [00:39,  3.28s/it]


12it [00:42,  3.16s/it]


12it [00:42,  3.52s/it]

Hard dataset evaluation complete.


In [6]:
# Pull results from the hard experiment
hard_dataset_obj = next(d for d in client.list_datasets() if d.name == HARD_DATASET_NAME)
hard_experiments = list(client.list_projects(reference_dataset_id=hard_dataset_obj.id))
hard_exp = next(e for e in hard_experiments if e.name.startswith("gpt-4o-mini-hard-"))
hard_runs = list(client.list_runs(project_id=hard_exp.id, execution_order=1))
print(f"Loaded {len(hard_runs)} runs from: {hard_exp.name}")

hard_records = []
for r in hard_runs:
    inp          = r.inputs.get("inputs", r.inputs)
    correctness  = None
    actionability = None
    if r.feedback_stats:
        if "correctness" in r.feedback_stats:
            correctness = r.feedback_stats["correctness"]["avg"]
        if "actionability" in r.feedback_stats:
            actionability = r.feedback_stats["actionability"]["avg"]
    hard_records.append({
        "question":     inp.get("question", "")[:70],
        "category":     inp.get("category", "unknown"),
        "correctness":  correctness,
        "actionability": actionability,
    })

df_hard = pd.DataFrame(hard_records)
df_hard["correctness"]  = pd.to_numeric(df_hard["correctness"],  errors="coerce")
df_hard["actionability"] = pd.to_numeric(df_hard["actionability"], errors="coerce")

# ── Data completeness check ───────────────────────────────────────────────
n_total           = len(df_hard)
n_correctness     = df_hard["correctness"].notna().sum()
n_actionability   = df_hard["actionability"].notna().sum()
n_missing         = n_total - n_correctness

print(f"\n=== Data Completeness ===")
print(f"Total runs          : {n_total}")
print(f"Correctness scores  : {n_correctness} / {n_total}  ({n_missing} missing — metrics below are on {n_correctness} examples)")
print(f"Actionability scores: {n_actionability} / {n_total}")

print("\n=== Hard Dataset — Aggregate Metrics ===")
print(f"Mean correctness  : {df_hard['correctness'].mean():.3f}  [n={n_correctness}]")
print(f"Mean actionability: {df_hard['actionability'].mean():.3f}  [n={n_actionability}]")
print(f"Pass rate correctness (>=0.8): {(df_hard['correctness'] >= 0.8).mean()*100:.0f}%")
print(f"Fail rate correctness (<0.5) : {(df_hard['correctness'] < 0.5).mean()*100:.0f}%")

print("\n=== Performance by Category ===")
print(df_hard.groupby("category")[["correctness", "actionability"]].agg(["mean", "count"]).round(3))

print("\n=== Lowest scoring examples (correctness) ===")
print(df_hard.nsmallest(5, "correctness")[["question", "correctness", "actionability", "category"]].to_string(index=False))


Loaded 12 runs from: gpt-4o-mini-hard-9af36d57

=== Data Completeness ===
Total runs          : 12
Correctness scores  : 7 / 12  (5 missing — metrics below are on 7 examples)
Actionability scores: 6 / 12

=== Hard Dataset — Aggregate Metrics ===
Mean correctness  : 0.429  [n=7]
Mean actionability: 0.750  [n=6]
Pass rate correctness (>=0.8): 25%
Fail rate correctness (<0.5) : 33%

=== Performance by Category ===
                  correctness       actionability      
                         mean count          mean count
category                                               
Ambiguous                 0.0     1           1.0     1
Conflicting               1.0     2           0.7     2
Insufficient data         0.0     1           0.7     1
Multi-step                0.0     2           1.0     1
Trick                     1.0     1           0.4     1

=== Lowest scoring examples (correctness) ===
                                                              question  correctness  actio

---
> **Note on data completeness:** 5 of 12 runs are missing correctness scores and 6 of 12 are missing actionability scores — visible in the counts above.
> This happens when `feedback_stats` is not yet populated at pull time (e.g. the evaluator run is still in flight or the child run hasn't propagated).
> All aggregate metrics are computed only on examples that have scores; categories with a single scored example (Ambiguous, Insufficient data, Trick) should be interpreted cautiously.


In [7]:
# ── Actionability analysis ────────────────────────────────────────────────
df_action = df_hard.dropna(subset=["actionability"])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Actionability scores — hard dataset", fontsize=13, fontweight="bold")

# By category
cat_action = df_action.groupby("category")["actionability"].mean().sort_values()
colors = sns.color_palette("husl", len(cat_action))
axes[0].barh(cat_action.index, cat_action.values, color=colors)
axes[0].axvline(0.8, color="green", linestyle="--", label="Target (0.8)")
axes[0].set_xlabel("Mean Actionability Score")
axes[0].set_title("Actionability by category")
axes[0].set_xlim(0, 1.1)
axes[0].legend()

# Correctness vs actionability scatter
df_both = df_hard.dropna(subset=["correctness", "actionability"])
cat_palette = {c: col for c, col in zip(df_both["category"].unique(),
                                         sns.color_palette("husl", df_both["category"].nunique()))}
for cat, grp in df_both.groupby("category"):
    axes[1].scatter(grp["correctness"], grp["actionability"],
                    label=cat, color=cat_palette[cat], s=100, alpha=0.8)
axes[1].set_xlabel("Correctness score")
axes[1].set_ylabel("Actionability score")
axes[1].set_title("Correctness vs Actionability")
axes[1].axhline(0.8, color="green", linestyle="--", alpha=0.5)
axes[1].axvline(0.8, color="green", linestyle="--", alpha=0.5)
axes[1].set_xlim(-0.1, 1.1)
axes[1].set_ylim(-0.1, 1.1)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("actionability_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: actionability_analysis.png")

print("\n=== Actionability summary by category ===")
print(df_action.groupby("category")["actionability"].agg(["mean", "min", "max", "count"]).round(3))

print("\n=== Correctness vs Actionability correlation ===")
if len(df_both) >= 3:
    corr = df_both["correctness"].corr(df_both["actionability"])
    print(f"Pearson r = {corr:.3f}  [n={len(df_both)}]")
    if corr > 0.5:
        print("→ Answers that are more correct tend to also be more actionable.")
    elif corr < -0.2:
        print("→ Surprisingly, more correct answers are not necessarily more actionable.")
    else:
        print("→ Weak correlation: correctness and actionability are partially independent dimensions.")
else:
    print("Too few paired observations for reliable correlation.")


Saved: actionability_analysis.png

=== Actionability summary by category ===
                   mean  min  max  count
category                                
Ambiguous           1.0  1.0  1.0      1
Conflicting         0.7  0.7  0.7      2
Insufficient data   0.7  0.7  0.7      1
Multi-step          1.0  1.0  1.0      1
Trick               0.4  0.4  0.4      1

=== Correctness vs Actionability correlation ===
Pearson r = -0.728  [n=6]
→ Surprisingly, more correct answers are not necessarily more actionable.


/var/folders/gh/r4_2cb497nl1c31dzl76npj80000gp/T/ipykernel_45374/2099415345.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
### Comparison: original dataset vs hard dataset

The chart below shows whether difficulty actually changes the score.
If the evaluation framework is working, the hard dataset score should be lower.


In [8]:
# Pull original dataset results for comparison
ORIG_DATASET_NAME = "marketing-analytics-qa-DinaBB"
orig_dataset_obj  = next(d for d in client.list_datasets() if d.name == ORIG_DATASET_NAME)
orig_experiments  = list(client.list_projects(reference_dataset_id=orig_dataset_obj.id))
orig_exp          = next(e for e in orig_experiments if e.name.startswith("gpt-4o-mini-")
                         and "hard" not in e.name and "dual" not in e.name)
orig_runs         = list(client.list_runs(project_id=orig_exp.id, execution_order=1))

orig_scores = []
for r in orig_runs:
    if r.feedback_stats and "correctness" in r.feedback_stats:
        orig_scores.append(r.feedback_stats["correctness"]["avg"])

orig_mean = pd.Series(orig_scores).mean()
hard_mean = df_hard["correctness"].mean()

# ── Side-by-side bar chart ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Does dataset difficulty change model scores?", fontsize=13, fontweight="bold")

# Overall comparison
bars = axes[0].bar(
    ["Original\n(20 easy)", "Hard\n(12 stress-test)"],
    [orig_mean, hard_mean],
    color=["#3498db", "#e74c3c"],
    width=0.5,
)
for bar, val in zip(bars, [orig_mean, hard_mean]):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.01, f"{val:.2f}",
                 ha="center", va="bottom", fontweight="bold", fontsize=13)
axes[0].axhline(0.8, linestyle="--", color="green", label="Pass threshold (0.8)")
axes[0].set_ylim(0, 1.15)
axes[0].set_ylabel("Mean Correctness Score (gpt-4o-mini)")
axes[0].set_title("Overall mean correctness")
axes[0].legend()

# Hard dataset by category
cat_means = df_hard.groupby("category")["correctness"].mean().sort_values()
colors = sns.color_palette("husl", len(cat_means))
axes[1].barh(cat_means.index, cat_means.values, color=colors)
axes[1].axvline(0.8, color="green", linestyle="--", label="Pass threshold (0.8)")
axes[1].set_xlabel("Mean Correctness Score")
axes[1].set_title("Hard dataset — score by category")
axes[1].legend()

plt.tight_layout()
plt.savefig("hard_vs_easy_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: hard_vs_easy_comparison.png")


Saved: hard_vs_easy_comparison.png


/var/folders/gh/r4_2cb497nl1c31dzl76npj80000gp/T/ipykernel_45374/3360766624.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
### What these results show

#### The evaluation is working

A score drop on the hard dataset confirms that the original ceiling result was a dataset
problem, not a model capability ceiling. The evaluation framework; target function,
LLM-as-judge, LangSmith experiment tracking, is sound. What needed to change was the
questions.

#### Where the model struggles

The category breakdown reveals the specific failure modes:

- **Multi-step reasoning (0.0)**: the model scored 0.0 on both multi-step questions. It tends
  to give a direct allocation or recommendation without working through the prerequisite
  diagnostic checks (e.g. diminishing returns, audience saturation, week-over-week trends).
  This is the most significant failure mode in the hard dataset.
- **Insufficient data (0.0)**: the model scored 0.0 on the insufficient-data category. Rather
  than flagging that a sample of 4 or 6 conversions is too small to conclude anything, it
  tends to provide a recommendation anyway — rewarding confidence over statistical honesty.
- **Ambiguous (0.0–0.5)**: the model sometimes acknowledges context dependency correctly,
  but other times gives a single confident answer where the right response is to state what
  the answer depends on before committing.

#### Where the model performed well

- **Trick / premise challenge (1.0)**: the model correctly identified and pushed back on
  flawed assumptions (e.g. evaluating an awareness campaign on ROAS, or treating
  Bonferroni correction as proof of equivalence). This was not a failure mode.
- **Conflicting signals (1.0)**: the model correctly diagnosed conflicts between CPA and
  ROAS and recommended the right investigative path rather than blindly optimising one metric.

#### The actionability–correctness tension (r = −0.73)

A surprising finding from the dual evaluator setup: correctness and actionability are
**negatively correlated** (Pearson r = −0.728). Categories that score high on correctness
(Trick, Conflicting) tend to score lower on actionability, and vice versa.

This makes sense on reflection: the *correct* answer to a nuanced question is often
*"it depends"* or *"your sample is too small"*, which the actionability judge penalises
for being non-committal. Meanwhile, an overconfident but wrong answer can sound highly
actionable. This is a core limitation of using a single LLM-as-judge for both dimensions
simultaneously; **correctness and actionability pull in opposite directions on hard questions**.
For deployment decisions, both dimensions are needed and should be reported separately.

#### What this means for prompt design

The failure modes are fixable without a model upgrade. The system prompt can be
extended with explicit instructions:

- *"If a question requires multiple reasoning steps, work through each step before giving a recommendation."*
- *"If the data described is insufficient to support a conclusion, say so explicitly before offering any guidance."*
- *"If the answer depends on context, state what it depends on before answering."*

A re-run of the hard dataset with an updated system prompt would show whether those
instructions close the gap; that is the next experiment.

#### Client-facing conclusion

> *The easy dataset told us the model knows its definitions. The hard dataset tells us
> where it overcommits or skips steps: multi-step reasoning and small-sample caution are
> the two areas that need targeted prompt engineering before deployment.
> The negative correlation between correctness and actionability (r = −0.73) is an additional
> finding: on hard questions, these two dimensions trade off against each other and must be
> tracked separately in any production evaluation pipeline.*
